In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
lemma = WordNetLemmatizer()
ps = PorterStemmer()
import re


from scipy.spatial import distance
from scipy.spatial import minkowski_distance
from scipy.spatial.distance import cosine



[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sophie/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#Electronics Dataset:

import numpy as np
import pandas as pd
fileElectr='amazon_reviews_us_Electronics_v1_00.tsv'
df=pd.read_csv(fileElectr, sep="\t", header=0, on_bad_lines='skip')
df=df.dropna(subset=['review_headline', 'review_body', 'star_rating'])

In [17]:
from transformers import pipeline


def emotion_distilbert(df_sample, column_name):
    classifier = pipeline("text-classification",model='bhadresh-savani/distilbert-base-uncased-emotion', return_all_scores=True)

    for i in range (0, len(df_sample[column_name])):

        #processed text
        text=df_sample[column_name][i]  
        if len(text) > 512:
            text = text[:512]
    
        prediction = classifier(text )

        for j in range(0,6):  #loop over the six emotions:
            df_sample.loc[i, ("distilbert_"+column_name+prediction[0][j]['label'])]=prediction[0][j]['score'] #BP=body 

    return df_sample
    
    
    
    
    

def emotion_roberta(df_sample, column_name):
    classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)

    for i in range (0, len(df_sample[column_name])):

        #processed text
        text=df_sample[column_name][i]  

        if len(text) > 512:
            text = text[:512]

        prediction = classifier(text )

        for j in range(0,7):  #loop over the  emotions:
            df_sample.loc[i, ("roberta_"+column_name+prediction[0][j]['label'])]=prediction[0][j]['score'] #BP=body processed


    return df_sample



In [7]:
def text_process2(reviews, column_name):  #input is the dataframe
    for i  in range(0, reviews[column_name].count()):
       review_body=reviews.loc[i, (column_name)]  #tokens= word_tokenize(df_sample.loc[1, ('review_body')])
       review_body=re.sub('<br\s?\/>|<br>', " ", review_body)  #remove the br
       tokens= word_tokenize(review_body)
       tokens = [w.lower()  for w in tokens ]
       #tokens = [w for w in tokens if not w in stop_words]
       tokens = [w for w in tokens if w.isalpha()] #remove non alphabetic items like like 5 or ;
       tokens = [lemma.lemmatize(w) for w in tokens]
       # tokens = [ps.stem(w) for w in tokens]
       column_name_out=column_name+"_processed"
       reviews.loc[i, (column_name_out)]=' '.join(tokens)
       
       if i%10000==0:
           print ("\n text_process:   We are at i=", str(i))
       
    return reviews




In [5]:
n_samples=500

N_rewiews=df.loc[df['star_rating'] == 1].sample(n_samples, replace=False, random_state=1900)
N_rewiew2=df.loc[df['star_rating'] == 5].sample(n_samples, replace=False, random_state=1900)


samplesize=n_samples*2

N_rewiews=N_rewiews.append(N_rewiew2)


/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_15852/1580433119.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_rewiews=N_rewiews.append(N_rewiew2)


In [9]:
N_rewiews=N_rewiews.reset_index()

In [10]:
N_rewiews=text_process2(N_rewiews,'review_body')
N_rewiews=text_process2(N_rewiews,'review_headline')






 text_process:   We are at i= 0

 text_process:   We are at i= 0


In [11]:
N_rewiews.head(1)

,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date,review_body_processed,review_headline_processed
0,1677873,US,18585476,R31LJGNJRWGRS,B003ARSOWQ,864558418,Timex T715BW3 Dual Alarm Clock Radio (Black),Electronics,1,0,1,N,Y,"Does it keep time, not really.","I have had the product for just under a month,...",2013-12-14,i have had the product for just under a month ...,doe it keep time not really


In [18]:
emotion_roberta(N_rewiews, 'review_body')
emotion_distilbert(N_rewiews, 'review_body')
emotion_roberta(N_rewiews, 'review_body_processed')
emotion_distilbert(N_rewiews, 'review_body_processed')


emotion_roberta(N_rewiews, 'review_headline')
emotion_distilbert(N_rewiews, 'review_headline')
emotion_roberta(N_rewiews, 'review_headline_processed')
emotion_distilbert(N_rewiews, 'review_headline_processed')




,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,...,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise,distilbert_review_headline_processedsadness,distilbert_review_headline_processedjoy,distilbert_review_headline_processedlove,distilbert_review_headline_processedanger,distilbert_review_headline_processedfear,distilbert_review_headline_processedsurprise
0,1677873,US,18585476,R31LJGNJRWGRS,B003ARSOWQ,864558418,Timex T715BW3 Dual Alarm Clock Radio (Black),Electronics,1,0,...,0.007847,0.839019,0.079229,0.038343,0.024349,0.210709,0.004549,0.680612,0.076367,0.003413
1,1856468,US,51075252,R35YF0DWJE87A4,B0044WS7KK,471031907,"Aerial7 Perisher - Black - black, one size",Electronics,1,0,...,0.002500,0.122202,0.713169,0.024730,0.997363,0.000588,0.000212,0.001171,0.000417,0.000248
2,600712,US,15700552,R2PQJHUI1SF24E,B00INO6JX2,703104763,Samsung SSG-5150GB 3D Active Glasses,Electronics,1,1,...,0.004079,0.698276,0.124367,0.081975,0.010798,0.002448,0.000436,0.973386,0.012356,0.000575
3,2871105,US,26397372,R1B30LZ8X5V07Q,B00008VSK5,50332,Acoustic Research MS805 Adaptatip Flex Pin,Electronics,1,4,...,0.006795,0.329728,0.535445,0.063412,0.963765,0.005869,0.000767,0.015357,0.013339,0.000904
4,1381712,US,20903669,R2B9JFV8C5AHX,B007N16IYG,623454097,Panasonic Deep Base Ergo-Fit Inner Ear Earbud ...,Electronics,1,1,...,0.003589,0.608421,0.278068,0.037570,0.994012,0.001515,0.000490,0.003532,0.000267,0.000184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,972872,US,38833858,R37XLSLYAXFM75,B002L6HDQA,498454528,Apple iPod nano 8 GB Purple,Electronics,5,0,...,0.472787,0.424965,0.012054,0.065021,0.020893,0.831073,0.007958,0.097601,0.032997,0.009478
996,147402,US,943239,R1FNDRUXBLAEU6,B003QN87AI,905030210,NEW 15 FT / 4.6 M High Speed HDMI Cable M/M 1....,Electronics,5,0,...,0.472787,0.424965,0.012054,0.065021,0.020893,0.831073,0.007958,0.097601,0.032997,0.009478
997,2425429,US,35158468,R3L4N9YXU9TCM2,B002MAPS6W,808084969,Clip Plus 4 GB MP3 Player (Black),Electronics,5,1,...,0.101398,0.371140,0.071178,0.442030,0.323867,0.310216,0.015452,0.082875,0.244135,0.023455
998,871753,US,20217103,R2U0FI8C8WQX9A,B005LJQNQU,706584662,BlueRigger Digital Optical Audio Toslink Cable...,Electronics,5,0,...,0.337709,0.415376,0.009322,0.227672,0.001373,0.993674,0.000756,0.002275,0.001322,0.000601


In [21]:
print(N_rewiews.review_body[14] )

print("\n" , N_rewiews.review_headline[14] )
N_rewiews.iloc[14]



Price almost tripled when bill came.  This is not needed with the camera scanner. Power comes through the computer connection.

 Over priced & not needed !!!!!


index                                                  2861885
marketplace                                                 US
customer_id                                           13817721
review_id                                       R3NJ72RXK0J6JP
product_id                                          B00009UTWB
                                                     ...      
distilbert_review_headline_processedjoy               0.388836
distilbert_review_headline_processedlove              0.005196
distilbert_review_headline_processedanger             0.463907
distilbert_review_headline_processedfear              0.018996
distilbert_review_headline_processedsurprise          0.004222
Name: 14, Length: 70, dtype: object

In [22]:
N_rewiews.columns

Index(['index', 'marketplace', 'customer_id', 'review_id', 'product_id',
       'product_parent', 'product_title', 'product_category', 'star_rating',
       'helpful_votes', 'total_votes', 'vine', 'verified_purchase',
       'review_headline', 'review_body', 'review_date',
       'review_body_processed', 'review_headline_processed',
       'roberta_review_bodyanger', 'roberta_review_bodydisgust',
       'roberta_review_bodyfear', 'roberta_review_bodyjoy',
       'roberta_review_bodyneutral', 'roberta_review_bodysadness',
       'roberta_review_bodysurprise', 'distilbert_review_bodysadness',
       'distilbert_review_bodyjoy', 'distilbert_review_bodylove',
       'distilbert_review_bodyanger', 'distilbert_review_bodyfear',
       'distilbert_review_bodysurprise', 'roberta_review_body_processedanger',
       'roberta_review_body_processeddisgust',
       'roberta_review_body_processedfear', 'roberta_review_body_processedjoy',
       'roberta_review_body_processedneutral',
       'roberta

In [ ]:
#distilbert (6): sadness, joy, love, anger, fear, surprise                                                            
#roberta (7): anger, disgust, fear, joy,neutral,sadness, surprise                                                          
#4*(6+7)=52


'roberta_review_body_processedanger',
'roberta_review_body_processeddisgust',
'roberta_review_body_processedfear',
'roberta_review_body_processedjoy',
'roberta_review_body_processedneutral',
'roberta_review_body_processedsadness',
'roberta_review_body_processedsurprise',



'roberta_review_bodyanger',
'roberta_review_bodydisgust',
'roberta_review_bodyfear',
'roberta_review_bodyjoy',
'roberta_review_bodyneutral',
'roberta_review_bodysadness',
'roberta_review_bodysurprise',



'roberta_review_headlineanger', 
'roberta_review_headlinedisgust',
'roberta_review_headlinefear', 
'roberta_review_headlinejoy',
'roberta_review_headlineneutral', 
'roberta_review_headlinesadness',
'roberta_review_headlinesurprise',





'roberta_review_headline_processedanger', 
'roberta_review_headline_processeddisgust',
'roberta_review_headline_processedfear', 
'roberta_review_headline_processedjoy',
'roberta_review_headline_processedneutral', 
'roberta_review_headline_processedsadness',
'roberta_review_headline_processedsurprise',



'distilbert_review_body_processedsadness',
'distilbert_review_body_processedjoy',
'distilbert_review_body_processedlove',
'distilbert_review_body_processedanger',
'distilbert_review_body_processedfear',
'distilbert_review_body_processedsurprise',


'distilbert_review_bodysadness',
'distilbert_review_bodyjoy',
'distilbert_review_bodylove',
'distilbert_review_bodyanger', 
'distilbert_review_bodyfear',
'distilbert_review_bodysurprise',



'distilbert_review_headlinesadness',
'distilbert_review_headlinejoy',
'distilbert_review_headlinelove',
'distilbert_review_headlineanger', 
'distilbert_review_headlinefear',
 'distilbert_review_headlinesurprise',



'distilbert_review_headline_processedsadness',
'distilbert_review_headline_processedjoy',
 'distilbert_review_headline_processedlove',
'distilbert_review_headline_processedanger', 
'distilbert_review_headline_processedfear',
 'distilbert_review_headline_processedsurprise',



In [25]:
#Only on body  (UNPROCESSED)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix


df_sample_all_test=N_rewiews[[  
'roberta_review_bodyanger',
'roberta_review_bodydisgust',
'roberta_review_bodyfear',
'roberta_review_bodyjoy',
'roberta_review_bodyneutral',
'roberta_review_bodysadness',
'roberta_review_bodysurprise',

'distilbert_review_bodysadness',
'distilbert_review_bodyjoy',
'distilbert_review_bodylove',
'distilbert_review_bodyanger', 
'distilbert_review_bodyfear',
'distilbert_review_bodysurprise',
    ]]




from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)


#'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}
clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))




target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive


print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))









0.852
                 precision    recall  f1-score   support

0 = rating of 1   0.872881  0.824000  0.847737       125
1 = rating of 5   0.833333  0.880000  0.856031       125

       accuracy                       0.852000       250
      macro avg   0.853107  0.852000  0.851884       250
   weighted avg   0.853107  0.852000  0.851884       250

[[103  22]
 [ 15 110]]


In [26]:
df_sample_all_test.shape

(1000, 13)

In [27]:
#Only on processed body 
df_sample_all_test=N_rewiews[[  

'roberta_review_body_processedanger',
'roberta_review_body_processeddisgust',
'roberta_review_body_processedfear',
'roberta_review_body_processedjoy',
'roberta_review_body_processedneutral',
'roberta_review_body_processedsadness',
'roberta_review_body_processedsurprise',

'distilbert_review_body_processedsadness',
'distilbert_review_body_processedjoy',
'distilbert_review_body_processedlove',
'distilbert_review_body_processedanger',
'distilbert_review_body_processedfear',
'distilbert_review_body_processedsurprise'
    ]]

X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))





0.78
                 precision    recall  f1-score   support

0 = rating of 1   0.791667  0.760000  0.775510       125
1 = rating of 5   0.769231  0.800000  0.784314       125

       accuracy                       0.780000       250
      macro avg   0.780449  0.780000  0.779912       250
   weighted avg   0.780449  0.780000  0.779912       250

[[ 95  30]
 [ 25 100]]


In [28]:
#only on UNprocessed healines

df_sample_all_test=N_rewiews[[  

'roberta_review_headlineanger', 
'roberta_review_headlinedisgust',
'roberta_review_headlinefear', 
'roberta_review_headlinejoy',
'roberta_review_headlineneutral', 
'roberta_review_headlinesadness',
'roberta_review_headlinesurprise',



'distilbert_review_headlinesadness',
'distilbert_review_headlinejoy',
'distilbert_review_headlinelove',
'distilbert_review_headlineanger', 
'distilbert_review_headlinefear',
 'distilbert_review_headlinesurprise',

    
    
    ]]

X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))








0.844
                 precision    recall  f1-score   support

0 = rating of 1   0.870690  0.808000  0.838174       125
1 = rating of 5   0.820896  0.880000  0.849421       125

       accuracy                       0.844000       250
      macro avg   0.845793  0.844000  0.843798       250
   weighted avg   0.845793  0.844000  0.843798       250

[[101  24]
 [ 15 110]]


In [29]:
#only on PROCESSED healines

df_sample_all_test=N_rewiews[[  
'distilbert_review_headline_processedsadness',
'distilbert_review_headline_processedjoy',
 'distilbert_review_headline_processedlove',
'distilbert_review_headline_processedanger', 
'distilbert_review_headline_processedfear',
 'distilbert_review_headline_processedsurprise',

'roberta_review_headline_processedanger', 
'roberta_review_headline_processeddisgust',
'roberta_review_headline_processedfear', 
'roberta_review_headline_processedjoy',
'roberta_review_headline_processedneutral', 
'roberta_review_headline_processedsadness',
'roberta_review_headline_processedsurprise',
    
    ]]

X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))






0.86
                 precision    recall  f1-score   support

0 = rating of 1   0.894737  0.816000  0.853556       125
1 = rating of 5   0.830882  0.904000  0.865900       125

       accuracy                       0.860000       250
      macro avg   0.862810  0.860000  0.859728       250
   weighted avg   0.862810  0.860000  0.859728       250

[[102  23]
 [ 12 113]]


In [ ]:
#PROCESSED headline and UNprocessed body text appear to perform better than the unprocessed headline and processed body text


In [30]:
#only on PROCESSED healines AND Distilbert only

#only on PROCESSED healines

df_sample_all_test=N_rewiews[[  
'distilbert_review_headline_processedsadness',
'distilbert_review_headline_processedjoy',
 'distilbert_review_headline_processedlove',
'distilbert_review_headline_processedanger', 
'distilbert_review_headline_processedfear',
 'distilbert_review_headline_processedsurprise',
    ]]

X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))



0.768
                 precision    recall  f1-score   support

0 = rating of 1   0.819048  0.688000  0.747826       125
1 = rating of 5   0.731034  0.848000  0.785185       125

       accuracy                       0.768000       250
      macro avg   0.775041  0.768000  0.766506       250
   weighted avg   0.775041  0.768000  0.766506       250

[[ 86  39]
 [ 19 106]]


In [31]:
#only on PROCESSED healines and Roberta only

df_sample_all_test=N_rewiews[[  

'roberta_review_headline_processedanger', 
'roberta_review_headline_processeddisgust',
'roberta_review_headline_processedfear', 
'roberta_review_headline_processedjoy',
'roberta_review_headline_processedneutral', 
'roberta_review_headline_processedsadness',
'roberta_review_headline_processedsurprise',
    
    ]]

X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))







0.856
                 precision    recall  f1-score   support

0 = rating of 1   0.932039  0.768000  0.842105       125
1 = rating of 5   0.802721  0.944000  0.867647       125

       accuracy                       0.856000       250
      macro avg   0.867380  0.856000  0.854876       250
   weighted avg   0.867380  0.856000  0.854876       250

[[ 96  29]
 [  7 118]]


In [ ]:
#THUS
# Roberta appears to be performing better than Distilbert on the PROCESSED healines





In [32]:
#UNprocessed  body and Distilbert only

df_sample_all_test=N_rewiews[[  

'distilbert_review_bodysadness',
'distilbert_review_bodyjoy',
'distilbert_review_bodylove',
'distilbert_review_bodyanger', 
'distilbert_review_bodyfear',
'distilbert_review_bodysurprise',
    ]]


X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))


0.708
                 precision    recall  f1-score   support

0 = rating of 1   0.802326  0.552000  0.654028       125
1 = rating of 5   0.658537  0.864000  0.747405       125

       accuracy                       0.708000       250
      macro avg   0.730431  0.708000  0.700717       250
   weighted avg   0.730431  0.708000  0.700717       250

[[ 69  56]
 [ 17 108]]


In [33]:
#UNprocessed  body and Roberta only

df_sample_all_test=N_rewiews[[  
'roberta_review_bodyanger',
'roberta_review_bodydisgust',
'roberta_review_bodyfear',
'roberta_review_bodyjoy',
'roberta_review_bodyneutral',
'roberta_review_bodysadness',
'roberta_review_bodysurprise',

    ]]


X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))



0.892
                 precision    recall  f1-score   support

0 = rating of 1   0.922414  0.856000  0.887967       125
1 = rating of 5   0.865672  0.928000  0.895753       125

       accuracy                       0.892000       250
      macro avg   0.894043  0.892000  0.891860       250
   weighted avg   0.894043  0.892000  0.891860       250

[[107  18]
 [  9 116]]


In [ ]:
#Roberta appears to be performing better on the UNprocessed  body

In [34]:
#DETAILED REVIEW for Roberta

Roberta_Body=['roberta_review_bodyanger',
'roberta_review_bodydisgust',
'roberta_review_bodyfear',
'roberta_review_bodyjoy',
'roberta_review_bodyneutral',
'roberta_review_bodysadness',
'roberta_review_bodysurprise']


Roberta_Body_Processed=[
'roberta_review_body_processedanger',
'roberta_review_body_processeddisgust',
'roberta_review_body_processedfear',
'roberta_review_body_processedjoy',
'roberta_review_body_processedneutral',
'roberta_review_body_processedsadness',
'roberta_review_body_processedsurprise' ] 

Roberta_Head=[
'roberta_review_headlineanger', 
'roberta_review_headlinedisgust',
'roberta_review_headlinefear', 
'roberta_review_headlinejoy',
'roberta_review_headlineneutral', 
'roberta_review_headlinesadness',
'roberta_review_headlinesurprise']


Roberta_Head_Processed=[
'roberta_review_headline_processedanger', 
'roberta_review_headline_processeddisgust',
'roberta_review_headline_processedfear', 
'roberta_review_headline_processedjoy',
'roberta_review_headline_processedneutral', 
'roberta_review_headline_processedsadness',
'roberta_review_headline_processedsurprise'] 








In [37]:
Roberta_Head_Processed+Roberta_Head

['roberta_review_headline_processedanger',
 'roberta_review_headline_processeddisgust',
 'roberta_review_headline_processedfear',
 'roberta_review_headline_processedjoy',
 'roberta_review_headline_processedneutral',
 'roberta_review_headline_processedsadness',
 'roberta_review_headline_processedsurprise',
 'roberta_review_headlineanger',
 'roberta_review_headlinedisgust',
 'roberta_review_headlinefear',
 'roberta_review_headlinejoy',
 'roberta_review_headlineneutral',
 'roberta_review_headlinesadness',
 'roberta_review_headlinesurprise']

In [38]:
df_sample_all_test=N_rewiews[ Roberta_Head_Processed+Roberta_Head ]

In [39]:
df_sample_all_test.head(1)

,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise,roberta_review_headlineanger,roberta_review_headlinedisgust,roberta_review_headlinefear,roberta_review_headlinejoy,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise
0,0.010669,0.01776,0.007133,0.007847,0.839019,0.079229,0.038343,0.023961,0.05149,0.005124,0.002476,0.683399,0.037004,0.196546


In [40]:
df_sample_all_test.shape #roberta incldues 7 emotions, so there should be 14 columns 

(1000, 14)

In [41]:
def run_SVC(df_sample_all_test,N_rewiews):

    X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

    clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)

    clf.score(X_test, y_test)
    y_test.value_counts()
    print(clf.score(X_test, y_test))

    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

    print(confusion_matrix(y_test, y_pred))



In [42]:
df_sample_all_test=N_rewiews[ Roberta_Body ] 
run_SVC(df_sample_all_test,N_rewiews)

0.892
                 precision    recall  f1-score   support

0 = rating of 1   0.922414  0.856000  0.887967       125
1 = rating of 5   0.865672  0.928000  0.895753       125

       accuracy                       0.892000       250
      macro avg   0.894043  0.892000  0.891860       250
   weighted avg   0.894043  0.892000  0.891860       250

[[107  18]
 [  9 116]]


In [43]:
df_sample_all_test=N_rewiews[ Roberta_Body_Processed ] 
run_SVC(df_sample_all_test,N_rewiews)

0.812
                 precision    recall  f1-score   support

0 = rating of 1   0.842105  0.768000  0.803347       125
1 = rating of 5   0.786765  0.856000  0.819923       125

       accuracy                       0.812000       250
      macro avg   0.814435  0.812000  0.811635       250
   weighted avg   0.814435  0.812000  0.811635       250

[[ 96  29]
 [ 18 107]]


In [44]:
df_sample_all_test=N_rewiews[ Roberta_Head ] 
run_SVC(df_sample_all_test,N_rewiews)

0.832
                 precision    recall  f1-score   support

0 = rating of 1   0.919192  0.728000  0.812500       125
1 = rating of 5   0.774834  0.936000  0.847826       125

       accuracy                       0.832000       250
      macro avg   0.847013  0.832000  0.830163       250
   weighted avg   0.847013  0.832000  0.830163       250

[[ 91  34]
 [  8 117]]


In [45]:
df_sample_all_test=N_rewiews[ Roberta_Head_Processed ] 
run_SVC(df_sample_all_test,N_rewiews)

0.856
                 precision    recall  f1-score   support

0 = rating of 1   0.932039  0.768000  0.842105       125
1 = rating of 5   0.802721  0.944000  0.867647       125

       accuracy                       0.856000       250
      macro avg   0.867380  0.856000  0.854876       250
   weighted avg   0.867380  0.856000  0.854876       250

[[ 96  29]
 [  7 118]]


In [46]:
df_sample_all_test=N_rewiews[ Roberta_Body_Processed +Roberta_Body  ] 
run_SVC(df_sample_all_test,N_rewiews)

0.864
                 precision    recall  f1-score   support

0 = rating of 1   0.882353  0.840000  0.860656       125
1 = rating of 5   0.847328  0.888000  0.867187       125

       accuracy                       0.864000       250
      macro avg   0.864841  0.864000  0.863922       250
   weighted avg   0.864841  0.864000  0.863922       250

[[105  20]
 [ 14 111]]


In [47]:
df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Head ] 
run_SVC(df_sample_all_test,N_rewiews)

0.896
                 precision    recall  f1-score   support

0 = rating of 1   0.923077  0.864000  0.892562       125
1 = rating of 5   0.872180  0.928000  0.899225       125

       accuracy                       0.896000       250
      macro avg   0.897629  0.896000  0.895893       250
   weighted avg   0.897629  0.896000  0.895893       250

[[108  17]
 [  9 116]]


In [48]:
df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Head_Processed ] 
run_SVC(df_sample_all_test,N_rewiews)

0.9
                 precision    recall  f1-score   support

0 = rating of 1   0.938596  0.856000  0.895397       125
1 = rating of 5   0.867647  0.944000  0.904215       125

       accuracy                       0.900000       250
      macro avg   0.903122  0.900000  0.899806       250
   weighted avg   0.903122  0.900000  0.899806       250

[[107  18]
 [  7 118]]


In [49]:
df_sample_all_test=N_rewiews[ Roberta_Body_Processed +Roberta_Head ] 
run_SVC(df_sample_all_test,N_rewiews)

0.884
                 precision    recall  f1-score   support

0 = rating of 1   0.887097  0.880000  0.883534       125
1 = rating of 5   0.880952  0.888000  0.884462       125

       accuracy                       0.884000       250
      macro avg   0.884025  0.884000  0.883998       250
   weighted avg   0.884025  0.884000  0.883998       250

[[110  15]
 [ 14 111]]


In [50]:
df_sample_all_test=N_rewiews[ Roberta_Body_Processed +Roberta_Head_Processed ] 
run_SVC(df_sample_all_test,N_rewiews)

0.872
                 precision    recall  f1-score   support

0 = rating of 1   0.911504  0.824000  0.865546       125
1 = rating of 5   0.839416  0.920000  0.877863       125

       accuracy                       0.872000       250
      macro avg   0.875460  0.872000  0.871704       250
   weighted avg   0.875460  0.872000  0.871704       250

[[103  22]
 [ 10 115]]


In [51]:
df_sample_all_test=N_rewiews[ Roberta_Head +Roberta_Head_Processed ] 
run_SVC(df_sample_all_test,N_rewiews)

0.864
                 precision    recall  f1-score   support

0 = rating of 1   0.902655  0.816000  0.857143       125
1 = rating of 5   0.832117  0.912000  0.870229       125

       accuracy                       0.864000       250
      macro avg   0.867386  0.864000  0.863686       250
   weighted avg   0.867386  0.864000  0.863686       250

[[102  23]
 [ 11 114]]


In [52]:
df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Body_Processed+Roberta_Head ] 
run_SVC(df_sample_all_test,N_rewiews)

0.904
                 precision    recall  f1-score   support

0 = rating of 1   0.917355  0.888000  0.902439       125
1 = rating of 5   0.891473  0.920000  0.905512       125

       accuracy                       0.904000       250
      macro avg   0.904414  0.904000  0.903975       250
   weighted avg   0.904414  0.904000  0.903975       250

[[111  14]
 [ 10 115]]


In [53]:
df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Body_Processed+Roberta_Head_Processed ] 
run_SVC(df_sample_all_test,N_rewiews)

0.904
                 precision    recall  f1-score   support

0 = rating of 1   0.924370  0.880000  0.901639       125
1 = rating of 5   0.885496  0.928000  0.906250       125

       accuracy                       0.904000       250
      macro avg   0.904933  0.904000  0.903945       250
   weighted avg   0.904933  0.904000  0.903945       250

[[110  15]
 [  9 116]]


In [54]:
df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 
run_SVC(df_sample_all_test,N_rewiews)

0.916
                 precision    recall  f1-score   support

0 = rating of 1   0.948276  0.880000  0.912863       125
1 = rating of 5   0.888060  0.952000  0.918919       125

       accuracy                       0.916000       250
      macro avg   0.918168  0.916000  0.915891       250
   weighted avg   0.918168  0.916000  0.915891       250

[[110  15]
 [  6 119]]


In [55]:
df_sample_all_test=N_rewiews[ Roberta_Body_Processed+Roberta_Head+Roberta_Head_Processed ] 
run_SVC(df_sample_all_test,N_rewiews)

0.884
                 precision    recall  f1-score   support

0 = rating of 1   0.921053  0.840000  0.878661       125
1 = rating of 5   0.852941  0.928000  0.888889       125

       accuracy                       0.884000       250
      macro avg   0.886997  0.884000  0.883775       250
   weighted avg   0.886997  0.884000  0.883775       250

[[105  20]
 [  9 116]]


In [56]:
df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Body_Processed+Roberta_Head+Roberta_Head_Processed ] 
run_SVC(df_sample_all_test,N_rewiews)

0.9
                 precision    recall  f1-score   support

0 = rating of 1   0.931034  0.864000  0.896266       125
1 = rating of 5   0.873134  0.936000  0.903475       125

       accuracy                       0.900000       250
      macro avg   0.902084  0.900000  0.899870       250
   weighted avg   0.902084  0.900000  0.899870       250

[[108  17]
 [  8 117]]


In [ ]:
#Thus the best performance appears to be Roberta_Body+Roberta_Head+Roberta_Head_Processed

In [57]:
#DETAILED REVIEW for Distilbert

Dist_Body=['distilbert_review_bodysadness',
'distilbert_review_bodyjoy',
'distilbert_review_bodylove',
'distilbert_review_bodyanger', 
'distilbert_review_bodyfear',
'distilbert_review_bodysurprise']


Dist_Body_Processed=[
'distilbert_review_body_processedsadness',
'distilbert_review_body_processedjoy',
'distilbert_review_body_processedlove',
'distilbert_review_body_processedanger',
'distilbert_review_body_processedfear',
'distilbert_review_body_processedsurprise'  ] 

Dist_Head=[
'distilbert_review_headlinesadness',
'distilbert_review_headlinejoy',
'distilbert_review_headlinelove',
'distilbert_review_headlineanger', 
'distilbert_review_headlinefear',
 'distilbert_review_headlinesurprise']


Dist_Head_Processed=[
'distilbert_review_headline_processedsadness',
'distilbert_review_headline_processedjoy',
 'distilbert_review_headline_processedlove',
'distilbert_review_headline_processedanger', 
'distilbert_review_headline_processedfear',
 'distilbert_review_headline_processedsurprise'] 



In [58]:
df_sample_all_test=N_rewiews[Dist_Body] 
run_SVC(df_sample_all_test,N_rewiews)

0.708
                 precision    recall  f1-score   support

0 = rating of 1   0.802326  0.552000  0.654028       125
1 = rating of 5   0.658537  0.864000  0.747405       125

       accuracy                       0.708000       250
      macro avg   0.730431  0.708000  0.700717       250
   weighted avg   0.730431  0.708000  0.700717       250

[[ 69  56]
 [ 17 108]]


In [59]:
df_sample_all_test=N_rewiews[Dist_Body_Processed] 
run_SVC(df_sample_all_test,N_rewiews)

0.688
                 precision    recall  f1-score   support

0 = rating of 1   0.742268  0.576000  0.648649       125
1 = rating of 5   0.653595  0.800000  0.719424       125

       accuracy                       0.688000       250
      macro avg   0.697931  0.688000  0.684037       250
   weighted avg   0.697931  0.688000  0.684037       250

[[ 72  53]
 [ 25 100]]


In [60]:
df_sample_all_test=N_rewiews[Dist_Head] 
run_SVC(df_sample_all_test,N_rewiews)

0.764
                 precision    recall  f1-score   support

0 = rating of 1   0.817308  0.680000  0.742358       125
1 = rating of 5   0.726027  0.848000  0.782288       125

       accuracy                       0.764000       250
      macro avg   0.771668  0.764000  0.762323       250
   weighted avg   0.771668  0.764000  0.762323       250

[[ 85  40]
 [ 19 106]]


In [61]:
df_sample_all_test=N_rewiews[Dist_Head_Processed] 
run_SVC(df_sample_all_test,N_rewiews)

0.768
                 precision    recall  f1-score   support

0 = rating of 1   0.819048  0.688000  0.747826       125
1 = rating of 5   0.731034  0.848000  0.785185       125

       accuracy                       0.768000       250
      macro avg   0.775041  0.768000  0.766506       250
   weighted avg   0.775041  0.768000  0.766506       250

[[ 86  39]
 [ 19 106]]


In [62]:
df_sample_all_test=N_rewiews[Dist_Body+Dist_Body_Processed] 
run_SVC(df_sample_all_test,N_rewiews)

0.696
                 precision    recall  f1-score   support

0 = rating of 1   0.769231  0.560000  0.648148       125
1 = rating of 5   0.654088  0.832000  0.732394       125

       accuracy                       0.696000       250
      macro avg   0.711659  0.696000  0.690271       250
   weighted avg   0.711659  0.696000  0.690271       250

[[ 70  55]
 [ 21 104]]


In [63]:
df_sample_all_test=N_rewiews[Dist_Body+Dist_Head] 
run_SVC(df_sample_all_test,N_rewiews)

0.8
                 precision    recall  f1-score   support

0 = rating of 1   0.790698  0.816000  0.803150       125
1 = rating of 5   0.809917  0.784000  0.796748       125

       accuracy                       0.800000       250
      macro avg   0.800308  0.800000  0.799949       250
   weighted avg   0.800308  0.800000  0.799949       250

[[102  23]
 [ 27  98]]


In [64]:
df_sample_all_test=N_rewiews[Dist_Body+Dist_Head_Processed] 
run_SVC(df_sample_all_test,N_rewiews)

0.772
                 precision    recall  f1-score   support

0 = rating of 1   0.774194  0.768000  0.771084       125
1 = rating of 5   0.769841  0.776000  0.772908       125

       accuracy                       0.772000       250
      macro avg   0.772017  0.772000  0.771996       250
   weighted avg   0.772017  0.772000  0.771996       250

[[96 29]
 [28 97]]


In [65]:
df_sample_all_test=N_rewiews[Dist_Body_Processed+Dist_Head] 
run_SVC(df_sample_all_test,N_rewiews)

0.796
                 precision    recall  f1-score   support

0 = rating of 1   0.780303  0.824000  0.801556       125
1 = rating of 5   0.813559  0.768000  0.790123       125

       accuracy                       0.796000       250
      macro avg   0.796931  0.796000  0.795840       250
   weighted avg   0.796931  0.796000  0.795840       250

[[103  22]
 [ 29  96]]


In [66]:
df_sample_all_test=N_rewiews[Dist_Body_Processed+Dist_Head_Processed] 
run_SVC(df_sample_all_test,N_rewiews)

0.78
                 precision    recall  f1-score   support

0 = rating of 1   0.773438  0.792000  0.782609       125
1 = rating of 5   0.786885  0.768000  0.777328       125

       accuracy                       0.780000       250
      macro avg   0.780161  0.780000  0.779968       250
   weighted avg   0.780161  0.780000  0.779968       250

[[99 26]
 [29 96]]


In [67]:
df_sample_all_test=N_rewiews[Dist_Head+Dist_Head_Processed] 
run_SVC(df_sample_all_test,N_rewiews)

0.756
                 precision    recall  f1-score   support

0 = rating of 1   0.796296  0.688000  0.738197       125
1 = rating of 5   0.725352  0.824000  0.771536       125

       accuracy                       0.756000       250
      macro avg   0.760824  0.756000  0.754867       250
   weighted avg   0.760824  0.756000  0.754867       250

[[ 86  39]
 [ 22 103]]


In [68]:
df_sample_all_test=N_rewiews[Dist_Body+Dist_Body_Processed+Dist_Head] 
run_SVC(df_sample_all_test,N_rewiews)

0.792
                 precision    recall  f1-score   support

0 = rating of 1   0.792000  0.792000  0.792000       125
1 = rating of 5   0.792000  0.792000  0.792000       125

       accuracy                       0.792000       250
      macro avg   0.792000  0.792000  0.792000       250
   weighted avg   0.792000  0.792000  0.792000       250

[[99 26]
 [26 99]]


In [69]:
df_sample_all_test=N_rewiews[Dist_Body+Dist_Body_Processed+Dist_Head_Processed] 
run_SVC(df_sample_all_test,N_rewiews)

0.78
                 precision    recall  f1-score   support

0 = rating of 1   0.796610  0.752000  0.773663       125
1 = rating of 5   0.765152  0.808000  0.785992       125

       accuracy                       0.780000       250
      macro avg   0.780881  0.780000  0.779827       250
   weighted avg   0.780881  0.780000  0.779827       250

[[ 94  31]
 [ 24 101]]


In [70]:
df_sample_all_test=N_rewiews[Dist_Body+Dist_Head+Dist_Head_Processed] 
run_SVC(df_sample_all_test,N_rewiews)

0.8
                 precision    recall  f1-score   support

0 = rating of 1   0.795276  0.808000  0.801587       125
1 = rating of 5   0.804878  0.792000  0.798387       125

       accuracy                       0.800000       250
      macro avg   0.800077  0.800000  0.799987       250
   weighted avg   0.800077  0.800000  0.799987       250

[[101  24]
 [ 26  99]]


In [71]:
df_sample_all_test=N_rewiews[Dist_Body_Processed+Dist_Head+Dist_Head_Processed] 
run_SVC(df_sample_all_test,N_rewiews)

0.788
                 precision    recall  f1-score   support

0 = rating of 1   0.772727  0.816000  0.793774       125
1 = rating of 5   0.805085  0.760000  0.781893       125

       accuracy                       0.788000       250
      macro avg   0.788906  0.788000  0.787834       250
   weighted avg   0.788906  0.788000  0.787834       250

[[102  23]
 [ 30  95]]


In [72]:
df_sample_all_test=N_rewiews[Dist_Body+Dist_Body_Processed+Dist_Head+Dist_Head_Processed] 
run_SVC(df_sample_all_test,N_rewiews)

0.804
                 precision    recall  f1-score   support

0 = rating of 1   0.806452  0.800000  0.803213       125
1 = rating of 5   0.801587  0.808000  0.804781       125

       accuracy                       0.804000       250
      macro avg   0.804019  0.804000  0.803997       250
   weighted avg   0.804019  0.804000  0.803997       250

[[100  25]
 [ 24 101]]


In [ ]:
#Based on the above, comparing Distilbert and Roberta emotion scopres as input to an SVM to predict the sentiment:
# Roberta shows higher accuracies -- highest for body+head+head_p (0.916) --  while Distilbert performs worse.


In [ ]:
#Select the TOP performer and ADD EMBEDDINGS

In [76]:
df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 
run_SVC(df_sample_all_test,N_rewiews)

0.916
                 precision    recall  f1-score   support

0 = rating of 1   0.948276  0.880000  0.912863       125
1 = rating of 5   0.888060  0.952000  0.918919       125

       accuracy                       0.916000       250
      macro avg   0.918168  0.916000  0.915891       250
   weighted avg   0.918168  0.916000  0.915891       250

[[110  15]
 [  6 119]]


In [ ]:
#adding embeddings 

In [77]:
def df2emd(word2vec_model, N_rewiews, column_name):
    word2vec_model_embeddings = WordVecVectorizer(word2vec_model)

    word2vec_model_embeddings_ave_one_review_list=[]
    embed_only=pd.DataFrame()
    
    for i in range(0,len(N_rewiews[column_name]) ) : 
    #for i in range(0,len(N_rewiews['review_body_process']) ) : 
        #list_words=[N_rewiews['review_body_process'][i]]
        list_words=[N_rewiews[column_name][i]]

        list_words=check_against_word2vec_model(list_words, word2vec_model)
        word2vec_embeddings_one_review=word2vec_model_embeddings.transform(list_words)
        word2vec_model_embeddings_ave_one_review_list.append(word2vec_embeddings_one_review)

    embed_only=pd.DataFrame(np.concatenate(word2vec_model_embeddings_ave_one_review_list))



    #return word2vec_model_embeddings_ave_one_review_list
    
    return embed_only

In [81]:

class WordVecVectorizer(object):
    def __init__(self, word2vec_model):
        self.word2vec_model = word2vec_model
        self.dim = 300
    def transform(self, X):
        return np.array([
            np.mean([self.word2vec_model[w] for w in texts.split() if w in self.word2vec_model]
                    or [np.zeros(self.dim)], axis=0)
            for texts in X
        ])




def check_against_word2vec_model(list_topics, word2vec_model):
    for i  in range(0, len(list_topics) ):
       tokens= word_tokenize(list_topics[i])
       tokens = [w for w in tokens if w in word2vec_model.key_to_index ]
       list_topics[i]=' '.join(tokens)
       return list_topics






In [79]:
import gensim


file_embeddings_fast='crawl-300d-2M.vec'


word2vec_model_fast = gensim.models.KeyedVectors.load_word2vec_format(file_embeddings_fast) 
print(word2vec_model_fast.vector_size)




300


In [82]:
embed_only_fast_B=df2emd(word2vec_model_fast, N_rewiews, "review_body")
embed_only_fast_HP=df2emd(word2vec_model_fast, N_rewiews, "review_headline")

embed_combined=embed_only_fast_B.join(embed_only_fast_HP, lsuffix='_caller', rsuffix='_other')



In [83]:
embed_combined.shape

(1000, 600)

In [84]:
embed_combined

,0_caller,1_caller,2_caller,3_caller,4_caller,5_caller,6_caller,7_caller,8_caller,9_caller,...,290_other,291_other,292_other,293_other,294_other,295_other,296_other,297_other,298_other,299_other
0,-0.010403,-0.063589,-0.030153,-0.073472,0.025708,0.032458,-0.007578,0.044956,-0.040789,0.018031,...,-0.046375,0.040000,0.054738,-0.060075,0.036063,-0.156525,0.039750,-0.137062,-0.054550,0.018938
1,-0.017315,0.007123,-0.041455,-0.078637,-0.087770,0.033452,-0.058645,-0.031260,-0.046363,0.013523,...,-0.084700,-0.008533,-0.047967,-0.041167,-0.152000,0.031833,0.003667,0.141400,0.022133,0.114100
2,0.132667,-0.080967,0.039067,0.080700,-0.068233,0.189667,0.127867,-0.005567,-0.022567,0.002933,...,0.063400,0.116650,0.229600,-0.436600,0.020700,0.015950,0.026450,-0.007900,-0.153200,0.196150
3,-0.040769,0.019894,-0.007164,0.019969,0.028814,0.010154,-0.023325,0.056783,0.041658,0.024893,...,0.026667,0.078078,-0.011022,-0.026311,-0.097600,-0.029767,0.037800,-0.073644,-0.040656,-0.015200
4,-0.051654,0.054688,-0.000839,0.021699,-0.005643,-0.009996,-0.036498,-0.008047,0.067326,-0.019517,...,-0.012867,0.059511,0.044033,-0.058133,-0.096033,-0.112944,0.025500,0.006378,0.053533,0.073556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.087433,-0.111167,-0.089833,0.040033,-0.166833,0.087900,0.118133,0.029567,0.291500,-0.062067,...,-0.089300,-0.022200,0.018750,-0.528850,-0.371900,0.007050,0.043800,0.344350,0.074700,0.089100
996,-0.100500,-0.525100,0.301000,0.050600,-0.226300,-0.123200,-0.036700,0.058800,0.203900,0.117400,...,-0.089300,-0.022200,0.018750,-0.528850,-0.371900,0.007050,0.043800,0.344350,0.074700,0.089100
997,-0.017756,-0.068376,-0.029250,-0.030709,0.022804,0.006861,0.008717,0.058891,-0.018358,-0.007281,...,0.020850,-0.082300,-0.040900,-0.261325,-0.058275,0.134200,-0.018525,0.055800,-0.142575,-0.027725
998,-0.145981,0.082144,-0.032769,0.075531,-0.032931,0.074031,-0.066894,0.130319,0.054737,0.049287,...,-0.023100,-0.080100,0.144357,-0.190371,-0.114586,0.055400,0.032014,0.140600,-0.019300,-0.019043


In [85]:
#combine the embeddings with the emotions

embed_combined=embed_combined.join(df_sample_all_test, lsuffix='_caller', rsuffix='_other')

embed_combined

,0_caller,1_caller,2_caller,3_caller,4_caller,5_caller,6_caller,7_caller,8_caller,9_caller,...,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise
0,-0.010403,-0.063589,-0.030153,-0.073472,0.025708,0.032458,-0.007578,0.044956,-0.040789,0.018031,...,0.683399,0.037004,0.196546,0.010669,0.017760,0.007133,0.007847,0.839019,0.079229,0.038343
1,-0.017315,0.007123,-0.041455,-0.078637,-0.087770,0.033452,-0.058645,-0.031260,-0.046363,0.013523,...,0.407534,0.376775,0.032329,0.022509,0.095116,0.019774,0.002500,0.122202,0.713169,0.024730
2,0.132667,-0.080967,0.039067,0.080700,-0.068233,0.189667,0.127867,-0.005567,-0.022567,0.002933,...,0.031080,0.012691,0.002935,0.042930,0.037332,0.011042,0.004079,0.698276,0.124367,0.081975
3,-0.040769,0.019894,-0.007164,0.019969,0.028814,0.010154,-0.023325,0.056783,0.041658,0.024893,...,0.309574,0.564732,0.034902,0.031687,0.021541,0.011392,0.006795,0.329728,0.535445,0.063412
4,-0.051654,0.054688,-0.000839,0.021699,-0.005643,-0.009996,-0.036498,-0.008047,0.067326,-0.019517,...,0.426625,0.491464,0.024036,0.013662,0.043480,0.015211,0.003589,0.608421,0.278068,0.037570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.087433,-0.111167,-0.089833,0.040033,-0.166833,0.087900,0.118133,0.029567,0.291500,-0.062067,...,0.874900,0.009004,0.035351,0.017302,0.005385,0.002486,0.472787,0.424965,0.012054,0.065021
996,-0.100500,-0.525100,0.301000,0.050600,-0.226300,-0.123200,-0.036700,0.058800,0.203900,0.117400,...,0.874900,0.009004,0.035351,0.017302,0.005385,0.002486,0.472787,0.424965,0.012054,0.065021
997,-0.017756,-0.068376,-0.029250,-0.030709,0.022804,0.006861,0.008717,0.058891,-0.018358,-0.007281,...,0.768580,0.033544,0.144062,0.004054,0.002152,0.008048,0.101398,0.371140,0.071178,0.442030
998,-0.145981,0.082144,-0.032769,0.075531,-0.032931,0.074031,-0.066894,0.130319,0.054737,0.049287,...,0.782269,0.006615,0.082245,0.006042,0.001557,0.002321,0.337709,0.415376,0.009322,0.227672


In [86]:
run_SVC(embed_combined,N_rewiews)

0.928
                 precision    recall  f1-score   support

0 = rating of 1   0.949580  0.904000  0.926230       125
1 = rating of 5   0.908397  0.952000  0.929688       125

       accuracy                       0.928000       250
      macro avg   0.928988  0.928000  0.927959       250
   weighted avg   0.928988  0.928000  0.927959       250

[[113  12]
 [  6 119]]


In [87]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import make_classification



X_train, X_test, y_train, y_test = train_test_split(embed_combined, N_rewiews['star_rating'],  random_state=56)



clf = AdaBoostClassifier(n_estimators=50, random_state=156)
clf.fit(X_train, y_train)

y_pred=clf.predict(X_test)




print(clf.score(X_test, y_test))


target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive


print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))



0.936
                 precision    recall  f1-score   support

0 = rating of 1   0.965812  0.904000  0.933884       125
1 = rating of 5   0.909774  0.968000  0.937984       125

       accuracy                       0.936000       250
      macro avg   0.937793  0.936000  0.935934       250
   weighted avg   0.937793  0.936000  0.935934       250

[[113  12]
 [  4 121]]
